# Pipeline de datos masivos: contratación pública SECOP II, de la fuente al tablero
**Gestión de Datos Masivos G004 · Unidad 3 · I.U. Pascual Bravo** · Notebook de las **fases 2 a 8**, ejecutado de principio a fin sobre el stack Docker (Kafka + HDFS + PySpark).
Cada fase importa el módulo real de `pipeline/` (el mismo que usa `run_pipeline`) y explica en celdas de texto qué se decidió y por qué.

## Fase 1 · Fuente, pregunta y arquitectura
| | |
|---|---|
| **Fuente** | datos.gov.co (Socrata) · *SECOP II – Contratos Electrónicos* · dataset `jbjy-vk9h` (~5,7 M de registros; se actualiza a diario) |
| **Fecha** | `fecha_de_firma` |
| **id_registro** | `id_contrato` (único; verificado con `count(*) = count(distinct id_contrato)`) |
| **Pregunta** | ¿Cuánta contratación pública se adjudica de forma directa y en qué departamentos, modalidades y entidades se concentra el dinero? |
| **Usuarios** | veedurías ciudadanas, periodistas de datos y entes de control |
| **Arquitectura** | **Lambda**: capa batch (raw → curada → indicadores) + capa rápida (Structured Streaming) que se unen en el tablero |

```
API Socrata → [RAW JSONL por fecha_ingesta] → Kafka (3 part., key=id) ─┬→ consumidor → HDFS raw ─→ MapReduce ⇄ Spark
                                                                       │                      └→ PySpark → Parquet (curada, anio_mes) → calidad → SQLite (servicio) → tablero HTML
                                                                       └→ Structured Streaming (ventana 1 día, watermark, checkpoint) → SQLite (capa rápida)
```
**Por qué Lambda y no Kappa:** el batch permite reprocesar *todo* el historial desde el raw cuando cambian las reglas de limpieza y es la base de los indicadores auditables; la capa rápida solo cubre lo reciente con baja latencia. En Kappa todo pasaría por el flujo y la limpieza completa sería más difícil de rehacer.

In [1]:
import os, sys, json, sqlite3, subprocess
os.chdir("/app"); sys.path.insert(0, "/app")
import pandas as pd
pd.set_option("display.max_colwidth", 120, "display.width", 200)
from pipeline import config
print("Fuente :", config.SOCRATA_URL)
print("Ventana:", config.INGEST_DESDE, "→", config.INGEST_HASTA)
print("Kafka  :", config.KAFKA_BOOTSTRAP, "| tópico", config.TOPIC, "| particiones", config.PARTITIONS)
print("HDFS   :", config.HDFS_WEB)

Fuente : https://www.datos.gov.co/resource/jbjy-vk9h.json
Ventana: 2026-10-25T00:00:00 → 2026-11-05T00:00:00
Kafka  : kafka:9092 | tópico secop.contratos.raw | particiones 3
HDFS   : http://namenode:9870


## Fase 2 · Ingesta a la zona raw
`requests` con **paginación** (`$limit/$offset`, orden estable por `fecha_de_firma, id_contrato`) y **hasta 5 reintentos** con espera exponencial ante 429/5xx o errores de red.
Cada registro se guarda en un **sobre** `{fuente, dataset, id_registro, fecha_ingesta, payload}` con el registro original **intacto** en `payload`, en JSON Lines particionado por `fecha_ingesta=AAAA-MM-DD`.
Cada corrida deja un **manifiesto** (URL, parámetros, páginas, conteos, marca de agua).

In [2]:
from pipeline import f2_ingesta
manifiesto = f2_ingesta.ejecutar()
pd.Series({k: v for k, v in manifiesto.items() if k != "parametros"}, name="manifiesto")

2026-11-23 03:44:02,921 INFO [ingesta] ventana: fecha_de_firma >= '2026-10-25T00:00:00' AND fecha_de_firma < '2026-11-05T00:00:00'


2026-11-23 03:44:08,204 INFO [ingesta] página 1: 10000 registros (acumulado 10000)


2026-11-23 03:44:15,687 INFO [ingesta] página 2: 10000 registros (acumulado 20000)


2026-11-23 03:44:19,062 INFO [ingesta] página 3: 4464 registros (acumulado 24464)


2026-11-23 03:44:20,340 INFO [ingesta] manifiesto: {"run_id": "20261123T034402-0378f6", "fuente": "datos.gov.co", "dataset": "jbjy-vk9h", "url": "https://www.datos.gov.co/resource/jbjy-vk9h.json", "parametros": {"$where": "fecha_de_firma >= '2026-10-25T00:00:00' AND fecha_de_firma < '2026-11-05T00:00:00'", "$order": "fecha_de_firma,id_contrato", "$limit": 10000, "$offset": "0..N (paginado)"}, "fecha_ingesta": "2026-11-23", "paginas": 3, "registros_descargados": 24464, "registros_omitidos_por_marca": 0, "registros_escritos": 24464, "archivo": "/app/data/raw/datos.gov.co/jbjy-vk9h/fecha_ingesta=2026-11-23/part-20261123T034402-0378f6.jsonl", "marca_agua_antes": null, "marca_agua_despues": "2026-11-04T23:58:02.186", "segundos": 17.4, "finalizado": "2026-11-23T03:44:20Z"}


run_id                                                                                                   20261123T034402-0378f6
fuente                                                                                                             datos.gov.co
dataset                                                                                                               jbjy-vk9h
url                                                                            https://www.datos.gov.co/resource/jbjy-vk9h.json
fecha_ingesta                                                                                                        2026-11-23
paginas                                                                                                                       3
registros_descargados                                                                                                     24464
registros_omitidos_por_marca                                                                            

In [3]:
from pipeline.common import archivos_raw
linea = json.loads(open(archivos_raw()[0], encoding="utf-8").readline())
print("Sobre (payload resumido a 6 campos):")
print(json.dumps({**linea, "payload": {k: linea["payload"][k] for k in list(linea["payload"])[:6]} | {"...": f"{len(linea['payload'])} campos"}}, ensure_ascii=False, indent=2))

Sobre (payload resumido a 6 campos):
{
  "fuente": "datos.gov.co",
  "dataset": "jbjy-vk9h",
  "id_registro": "CO1.PCCNTR.10099677",
  "fecha_ingesta": "2026-11-23",
  "payload": {
    "nombre_entidad": "SUBRED INTEGRADA DE SERVICIOS DE SALUD SUR OCCIDENTE ESE.",
    "nit_entidad": "900959048",
    "departamento": "Distrito Capital de Bogotá",
    "ciudad": "No Definido",
    "localizaci_n": "Colombia, Bogotá, No Definido",
    "orden": "Territorial",
    "...": "87 campos"
  }
}


## Fase 3 · Mensajería con Kafka
Tópico `secop.contratos.raw` con **3 particiones**. El productor publica **cada línea del raw** con **key = id_registro** (un mismo contrato cae siempre en la misma partición) y es **idempotente** (`enable.idempotence`, `acks=all`). Un consumidor con **grupo** (`secop-hdfs-writer`) lee y confirma offsets *después* de escribir en HDFS.

**¿Qué pasa si el productor se ejecuta dos veces?** (1) Un *ledger* de archivos ya publicados evita republicar: la segunda corrida no envía nada. (2) Si alguien fuerza la republicación, Kafka sí duplica (es un log de eventos), pero la **zona curada deduplica por `id_registro`** y el streaming usa `dropDuplicates`; se demuestra con conteos en `evidencias/kafka_duplicados.txt`.

In [4]:
from pipeline import f3_kafka_producer, f3_kafka_consumer
print(f3_kafka_producer.ejecutar())

2026-11-23 03:44:20,491 INFO [productor] tópico secop.contratos.raw creado con 3 particiones


%4|1795405460.503|GETPID|rdkafka#producer-2| [thrd:main]: Failed to acquire idempotence PID from broker kafka:9092/1: Broker: Coordinator load in progress: retrying


2026-11-23 03:44:22,128 INFO [productor] part-20261123T034402-0378f6.jsonl: 24464 mensajes publicados


2026-11-23 03:44:22,129 INFO [productor] resumen: {'publicados': 24464, 'por_particion': {0: 8118, 1: 8154, 2: 8192}}


{'publicados': 24464, 'por_particion': {0: 8118, 1: 8154, 2: 8192}}


In [5]:
print("Segunda ejecución del productor (no debe publicar nada):")
print(f3_kafka_producer.ejecutar())

Segunda ejecución del productor (no debe publicar nada):
2026-11-23 03:44:22,156 INFO [productor] tópico secop.contratos.raw ya existe


2026-11-23 03:44:22,157 INFO [productor] nada nuevo que publicar (todos los archivos ya están en el ledger)


{'publicados': 0, 'por_particion': {}}


## Fase 4 · Data lake en HDFS y MapReduce
El consumidor guarda en `/datalake/raw/<fuente>/<dataset>/fecha_ingesta=…/` **un archivo por lote de 5.000 mensajes** (no uno por mensaje). Los lotes se alinean a offsets múltiplos de 5.000, así el nombre es determinista y reescribirlo es idempotente.

In [6]:
print(f3_kafka_consumer.ejecutar())
from pipeline.common import hdfs_client
cli = hdfs_client()
filas = [(d.replace(config.HDFS_RAW, "…"), f, cli.status(f"{d}/{f}")["length"]) for d, _, fs in cli.walk(config.HDFS_RAW) for f in fs]
pd.DataFrame(filas, columns=["carpeta", "archivo", "bytes"])

2026-11-23 03:44:26,404 INFO [consumidor] HDFS <- /datalake/raw/datos.gov.co/jbjy-vk9h/fecha_ingesta=2026-11-23/lote_p2_o0000000000-0000004999.jsonl (5000 mensajes)


2026-11-23 03:44:26,709 INFO [consumidor] HDFS <- /datalake/raw/datos.gov.co/jbjy-vk9h/fecha_ingesta=2026-11-23/lote_p1_o0000000000-0000004999.jsonl (5000 mensajes)


2026-11-23 03:44:27,311 INFO [consumidor] HDFS <- /datalake/raw/datos.gov.co/jbjy-vk9h/fecha_ingesta=2026-11-23/lote_p0_o0000000000-0000004999.jsonl (5000 mensajes)


2026-11-23 03:44:43,234 INFO [consumidor] HDFS <- /datalake/raw/datos.gov.co/jbjy-vk9h/fecha_ingesta=2026-11-23/lote_p2_o0000005000-0000008191.jsonl (3192 mensajes)


2026-11-23 03:44:43,327 INFO [consumidor] HDFS <- /datalake/raw/datos.gov.co/jbjy-vk9h/fecha_ingesta=2026-11-23/lote_p1_o0000005000-0000008153.jsonl (3154 mensajes)


2026-11-23 03:44:43,397 INFO [consumidor] HDFS <- /datalake/raw/datos.gov.co/jbjy-vk9h/fecha_ingesta=2026-11-23/lote_p0_o0000005000-0000008117.jsonl (3118 mensajes)


2026-11-23 03:44:43,419 INFO [consumidor] resumen: {'mensajes_leidos': 24464, 'archivos_escritos': 6}


{'mensajes_leidos': 24464, 'archivos_escritos': 6}


,carpeta,archivo,bytes
0,…/fecha_ingesta=2026-11-23,lote_p0_o0000000000-0000004999.jsonl,22139262
1,…/fecha_ingesta=2026-11-23,lote_p0_o0000005000-0000008117.jsonl,13803931
2,…/fecha_ingesta=2026-11-23,lote_p1_o0000000000-0000004999.jsonl,22168706
3,…/fecha_ingesta=2026-11-23,lote_p1_o0000005000-0000008153.jsonl,13965594
4,…/fecha_ingesta=2026-11-23,lote_p2_o0000000000-0000004999.jsonl,22133294
5,…/fecha_ingesta=2026-11-23,lote_p2_o0000005000-0000008191.jsonl,14133957


**MapReduce vs Spark.** Mapper y reducer en Python (flujo de Hadoop Streaming: `mapper | sort | reducer`) leen el raw directamente de HDFS y calculan **número de contratos y valor por departamento**; Spark hace el mismo cálculo y se comparan.
Primero se ejecuta como tubería `mapper | sort | reducer` leyendo HDFS (la que usa `run_pipeline`). Además se ejecuta como **job real de Hadoop Streaming** (`hadoop-streaming-3.2.1.jar` en el contenedor del NameNode, que trae Python 3; el stack no incluye YARN, así que usa el ejecutor local de MapReduce de Hadoop) con `bash run_evidencias.sh hadoop`, y su salida en HDFS (`/datalake/mapreduce_out`) se compara con Spark.

In [7]:
from pipeline import f4_mapreduce_vs_spark
_ = f4_mapreduce_vs_spark.ejecutar()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/23 03:44:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


departamento                                   MR contratos  Spark contr.             MR valor          Spark valor
Distrito Capital de Bogotá                             4525          4525    4,560,625,772,521    4,560,625,772,521
Antioquia                                              2313          2313    1,388,153,771,818    1,388,153,771,818
Valle del Cauca                                        2200          2200      131,851,924,130      131,851,924,130
Bolívar                                                1575          1575       76,911,514,521       76,911,514,521
Norte de Santander                                     1480          1480       49,141,472,011       49,141,472,011
Tolima                                                 1181          1181       49,426,192,748       49,426,192,748
Cundinamarca                                            947           947      112,900,636,105      112,900,636,105
Santander                                               921           92

In [ ]:
# Job real de Hadoop Streaming (se lanza desde el host: bash run_evidencias.sh hadoop)
ruta = "evidencias/hadoop_streaming_job.txt"
print(open(ruta, encoding="utf-8").read() if os.path.exists(ruta) else "Aún no se ha ejecutado: bash run_evidencias.sh hadoop")
print(open("evidencias/mapreduce_vs_spark_hadoop.txt", encoding="utf-8").read().splitlines()[-2:] if os.path.exists("evidencias/mapreduce_vs_spark_hadoop.txt") else "")

INFO mapreduce.Job: Job job_local1046237908_0001 running in uber mode : false
INFO mapreduce.Job: Job job_local1046237908_0001 completed successfully
contadores finales del job:
  Map input records=32707
  Map output records=32707
  Reduce input records=32707
  Reduce output records=34
--- hdfs dfs -ls /datalake/mapreduce_out
Found 2 items
-rw-r--r--   1 root supergroup          0 2026-11-23 04:14 /datalake/mapreduce_out/_SUCCESS
-rw-r--r--   1 root supergroup       1064 2026-11-23 04:14 /datalake/mapreduce_out/part-00000

['TOTAL contratos  MapReduce=32707  Spark=32707', 'Departamentos con diferencias: ninguno  ->  COINCIDEN']


## Fase 5 · Zona curada con PySpark
Lee de HDFS, **quita duplicados por `id_registro`**, corrige tipos, nulos y formatos, y guarda **Parquet particionado por `anio_mes`**. Cada decisión se mide (filas antes/después) y queda en `curada_stats.json`.
**Partición `anio_mes`:** las consultas son temporales; el mes da pocas particiones grandes (el día generaría archivos diminutos) y crece de forma predecible; `departamento` está muy sesgado hacia Bogotá/Valle/Antioquia.

In [8]:
from pipeline import f5_curada
st = f5_curada.ejecutar()
pd.DataFrame([{"problema": k, "qué se decidió": v["decision"], "por qué": v["por_que"],
               **{a: b for a, b in v.items() if a not in ("problema", "decision", "por_que")}} for k, v in st["decisiones"].items()]).set_index("problema").T

2026-11-23 03:45:00,937 INFO [curada] filas raw: 24464


2026-11-23 03:45:10,673 INFO [curada] curada: 24464 -> 24464 -> 24464 filas | JSONL 108.3 MB vs Parquet 1.7 MB


problema,duplicados,tipos,texto,descartes
qué se decidió,"Conservar una fila por id_registro: la de mayor fecha_ingesta y, en empate, la de mayor ultima_actualizacion.",fecha_* -> timestamp; valor_* -> double. Lo que no se pueda convertir queda NULL (no se inventa un valor).,Nulos disfrazados -> NULL (departamento -> 'Sin dato'); modalidad normalizada a un catálogo de 8 grupos; se crea es_...,Se descartan y se cuentan por razón en el reporte de calidad (no se borran en silencio; el raw conserva todo).
por qué,El raw es un historial; la versión más reciente refleja el estado actual del contrato y evita contar dos veces el di...,Las agregaciones por fecha y suma de dinero necesitan tipos reales; NULL explícito es auditable.,"Si no se unifican, la misma modalidad se cuenta como categorías distintas y el % de contratación directa se subestima.",Un solo valor absurdo distorsiona totales y promedios por departamento.
filas_antes,24464.0,NaN,NaN,24464.0
filas_despues,24464.0,NaN,NaN,24464.0
eliminadas,0.0,NaN,NaN,NaN
fecha_firma_nulas,NaN,0.0,NaN,NaN
valor_nulos,NaN,0.0,NaN,NaN
departamento_no_definido_antes,NaN,NaN,732.0,NaN
modalidades_distintas_antes,NaN,NaN,12.0,NaN
modalidades_distintas_despues,NaN,NaN,8.0,NaN


In [9]:
t = st["tamano"]
print(f"JSONL raw en HDFS : {t['jsonl_raw_bytes']/1e6:8.1f} MB")
print(f"Parquet curada    : {t['parquet_bytes']/1e6:8.1f} MB   (−{t['reduccion_pct']} %)")
print("Descartes por razón:", st["descartes_por_razon"])
print("Partición:", st["particion"]["columna"], "→", st["particion"]["particiones"], "particiones")

JSONL raw en HDFS :    108.3 MB
Parquet curada    :      1.7 MB   (−98.5 %)
Descartes por razón: {'sin_fecha_firma': 0, 'valor_nulo': 0, 'valor_negativo': 0, 'valor_mayor_1e+13': 0}
Partición: anio_mes → 2 particiones


## Fase 6 · Calidad, carga incremental e idempotencia
**Validaciones** (4 críticas + 1 advertencia): conteo > 0, sin nulos en id/fecha/valor, sin `id_registro` repetidos, valor en [0, 1e13] y % de descartados < 5 %. Se guarda `reporte_calidad.json` con los registros descartados. **Si una crítica falla, el proceso termina con `exit 1` y `run_pipeline` se detiene.**

In [10]:
from pipeline import f6_calidad
rep = f6_calidad.ejecutar()
pd.DataFrame(rep["validaciones"])

2026-11-23 03:45:12,619 INFO [calidad] [OK] conteo_mayor_que_cero (crítica) -> 24464 filas


2026-11-23 03:45:12,620 INFO [calidad] [OK] sin_nulos_en_columnas_clave (crítica) -> {'id_registro': 0, 'fecha_de_firma': 0, 'valor_del_contrato': 0}


2026-11-23 03:45:12,621 INFO [calidad] [OK] sin_id_registro_repetidos (crítica) -> 0 repetidos


2026-11-23 03:45:12,622 INFO [calidad] [OK] valor_dentro_de_rango_0_a_1e13 (crítica) -> 0 fuera de rango


2026-11-23 03:45:12,622 INFO [calidad] [OK] descartados_menos_del_5_pct -> 0.00% descartado


,nombre,critica,ok,detalle
0,conteo_mayor_que_cero,True,True,24464 filas
1,sin_nulos_en_columnas_clave,True,True,"{'id_registro': 0, 'fecha_de_firma': 0, 'valor_del_contrato': 0}"
2,sin_id_registro_repetidos,True,True,0 repetidos
3,valor_dentro_de_rango_0_a_1e13,True,True,0 fuera de rango
4,descartados_menos_del_5_pct,False,True,0.00% descartado


**Carga incremental con marca de agua.** `data/state/watermark.json` guarda la última `fecha_de_firma` cargada (y los ids exactamente en esa marca, para no repetir el borde). Ampliamos la ventana hasta el 8-nov: solo debe traer lo nuevo; y repetirla debe traer 0.

In [11]:
print("Marca de agua actual:", json.load(open("data/state/watermark.json"))["fecha_de_firma"])
config.INGEST_HASTA = "2026-11-08T00:00:00"
m1 = f2_ingesta.ejecutar()
m2 = f2_ingesta.ejecutar()
pd.DataFrame([{k: m[k] for k in ("registros_descargados", "registros_omitidos_por_marca", "registros_escritos", "marca_agua_antes", "marca_agua_despues")} for m in (m1, m2)],
             index=["ampliar ventana (solo lo nuevo)", "misma ventana otra vez (0 nuevos)"])

Marca de agua actual: 2026-11-04T23:58:02.186
2026-11-23 03:45:12,709 INFO [ingesta] ventana: fecha_de_firma >= '2026-11-04T23:58:02.186' AND fecha_de_firma < '2026-11-08T00:00:00'


2026-11-23 03:45:33,415 INFO [ingesta] página 1: 8244 registros (acumulado 8243)


2026-11-23 03:45:33,886 INFO [ingesta] manifiesto: {"run_id": "20261123T034512-25cfa0", "fuente": "datos.gov.co", "dataset": "jbjy-vk9h", "url": "https://www.datos.gov.co/resource/jbjy-vk9h.json", "parametros": {"$where": "fecha_de_firma >= '2026-11-04T23:58:02.186' AND fecha_de_firma < '2026-11-08T00:00:00'", "$order": "fecha_de_firma,id_contrato", "$limit": 10000, "$offset": "0..N (paginado)"}, "fecha_ingesta": "2026-11-23", "paginas": 1, "registros_descargados": 8244, "registros_omitidos_por_marca": 1, "registros_escritos": 8243, "archivo": "/app/data/raw/datos.gov.co/jbjy-vk9h/fecha_ingesta=2026-11-23/part-20261123T034512-25cfa0.jsonl", "marca_agua_antes": "2026-11-04T23:58:02.186", "marca_agua_despues": "2026-11-07T23:29:53.311", "segundos": 21.2, "finalizado": "2026-11-23T03:45:33Z"}


2026-11-23 03:45:33,907 INFO [ingesta] ventana: fecha_de_firma >= '2026-11-07T23:29:53.311' AND fecha_de_firma < '2026-11-08T00:00:00'


2026-11-23 03:45:35,023 INFO [ingesta] página 1: 1 registros (acumulado 0)


2026-11-23 03:45:35,024 INFO [ingesta] manifiesto: {"run_id": "20261123T034533-02a62f", "fuente": "datos.gov.co", "dataset": "jbjy-vk9h", "url": "https://www.datos.gov.co/resource/jbjy-vk9h.json", "parametros": {"$where": "fecha_de_firma >= '2026-11-07T23:29:53.311' AND fecha_de_firma < '2026-11-08T00:00:00'", "$order": "fecha_de_firma,id_contrato", "$limit": 10000, "$offset": "0..N (paginado)"}, "fecha_ingesta": "2026-11-23", "paginas": 1, "registros_descargados": 1, "registros_omitidos_por_marca": 1, "registros_escritos": 0, "archivo": null, "marca_agua_antes": "2026-11-07T23:29:53.311", "marca_agua_despues": "2026-11-07T23:29:53.311", "segundos": 1.1, "finalizado": "2026-11-23T03:45:35Z"}


,registros_descargados,registros_omitidos_por_marca,registros_escritos,marca_agua_antes,marca_agua_despues
ampliar ventana (solo lo nuevo),8244,1,8243,2026-11-04T23:58:02.186,2026-11-07T23:29:53.311
misma ventana otra vez (0 nuevos),1,1,0,2026-11-07T23:29:53.311,2026-11-07T23:29:53.311


**Idempotencia (demostración):** se corre el pipeline completo **dos veces** y se comparan los conteos de cada zona (raw, Kafka, HDFS, curada, servicio, streaming). Si alguno cambia, el resultado es `FALLÓ`. La curada se *reconstruye* desde el raw (`overwrite` de particiones) y el servicio usa `replace`, por eso repetir no duplica.

In [12]:
from pipeline import demo_idempotencia
try:
    demo_idempotencia.main()
except SystemExit as e:
    print("código de salida:", e.code)

2026-11-23 03:45:35,046 INFO [pipeline] === F2 ingesta raw (API -> JSONL) ===


2026-11-23 03:45:35,290 INFO [ingesta] ventana: fecha_de_firma >= '2026-11-07T23:29:53.311' AND fecha_de_firma < '2026-11-05T00:00:00'


2026-11-23 03:45:36,356 INFO [ingesta] página 1: 0 registros (acumulado 0)
2026-11-23 03:45:36,358 INFO [ingesta] manifiesto: {"run_id": "20261123T034535-8ab836", "fuente": "datos.gov.co", "dataset": "jbjy-vk9h", "url": "https://www.datos.gov.co/resource/jbjy-vk9h.json", "parametros": {"$where": "fecha_de_firma >= '2026-11-07T23:29:53.311' AND fecha_de_firma < '2026-11-05T00:00:00'", "$order": "fecha_de_firma,id_contrato", "$limit": 10000, "$offset": "0..N (paginado)"}, "fecha_ingesta": "2026-11-23", "paginas": 1, "registros_descargados": 0, "registros_omitidos_por_marca": 0, "registros_escritos": 0, "archivo": null, "marca_agua_antes": "2026-11-07T23:29:53.311", "marca_agua_despues": "2026-11-07T23:29:53.311", "segundos": 1.1, "finalizado": "2026-11-23T03:45:36Z"}
2026-11-23 03:45:36,405 INFO [pipeline] === F3 productor Kafka ===


2026-11-23 03:45:36,501 INFO [productor] tópico secop.contratos.raw ya existe


2026-11-23 03:45:37,007 INFO [productor] part-20261123T034512-25cfa0.jsonl: 8243 mensajes publicados
2026-11-23 03:45:37,007 INFO [productor] resumen: {'publicados': 8243, 'por_particion': {0: 2762, 1: 2750, 2: 2731}}
2026-11-23 03:45:37,018 INFO [pipeline] === F3/F4 consumidor Kafka -> HDFS ===


2026-11-23 03:45:37,924 INFO [consumidor] HDFS <- /datalake/raw/datos.gov.co/jbjy-vk9h/fecha_ingesta=2026-11-23/lote_p1_o0000008154-0000009999.jsonl (1846 mensajes)
2026-11-23 03:45:38,032 INFO [consumidor] HDFS <- /datalake/raw/datos.gov.co/jbjy-vk9h/fecha_ingesta=2026-11-23/lote_p2_o0000008192-0000009999.jsonl (1808 mensajes)


... (70 outputs left out)

2026-11-23 03:49:21,116 INFO [streaming] fin de la corrida: {'filas_entrada_ultima_corrida': 0, 'batch_id': 2}


2026-11-23 03:49:21,451 INFO [pipeline] PIPELINE COMPLETO SIN ERRORES


2026-11-23 03:49:22,042 INFO [idempotencia] conteos tras la corrida 2: {'raw_local_lineas': 32707, 'kafka_mensajes': 32707, 'hdfs_lineas': 32707, 'curada_filas(meta_corte)': 32707, 'batch_dias': 14, 'batch_contratos_en_serie': 32707, 'rapida_ventanas': 14, 'rapida_contratos': 32707}


zona / conteo                          corrida 1     corrida 2  igual
raw_local_lineas                           32707         32707  SI
kafka_mensajes                             32707         32707  SI
hdfs_lineas                                32707         32707  SI
curada_filas(meta_corte)                   32707         32707  SI
batch_dias                                    14            14  SI
batch_contratos_en_serie                   32707         32707  SI
rapida_ventanas                               14            14  SI
rapida_contratos                           32707         32707  SI

RESULTADO: IDEMPOTENTE - la segunda corrida no duplicó nada
código de salida: 0


## Fase 7 · Streaming: capa rápida (Lambda)
Spark **Structured Streaming** lee el tópico de Kafka y calcula **contratos y valor por ventana de 1 día** (event time = `fecha_de_firma`) con **watermark de 7 días** y **checkpoint**. `dropDuplicates(id_registro, event_time)` evita contar un mensaje republicado. La salida se *upserta* en SQLite (clave = ventana). Usa `Trigger.availableNow`: procesa lo pendiente y termina, lo que permite **reiniciar** y comprobar que no reprocesa ni duplica.

**Unión con el batch (Lambda):** el tablero usa la vista batch (curada, limpieza completa) y completa con la capa rápida los días posteriores al último día del batch. Cuando el batch se actualiza, la vista batch prevalece.

In [13]:
from pipeline import demo_streaming
try:
    demo_streaming.main()
except SystemExit as e:
    print("código de salida:", e.code)

:: loading settings :: url = jar:file:/usr/local/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml
2026-11-23 03:49:29,945 INFO [streaming] fin de la corrida: {'filas_entrada_ultima_corrida': 0, 'batch_id': 2}

Tras arranque #1: ventanas=14 contratos=32707


:: loading settings :: url = jar:file:/usr/local/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml
2026-11-23 03:49:38,995 INFO [streaming] fin de la corrida: {'filas_entrada_ultima_corrida': 0, 'batch_id': 2}

Tras arranque #2: ventanas=14 contratos=32707
RESULTADO: el reinicio no reprocesó ni duplicó
código de salida: 0


In [14]:
from pipeline.conteos import servicio
con = sqlite3.connect(config.SERVICIO_DB)
batch = pd.read_sql("SELECT fecha_firma AS fecha, contratos AS batch_contratos, valor_total AS batch_valor FROM ind_serie_diaria", con)
rapida = pd.read_sql("SELECT ventana_inicio AS fecha, contratos AS rapida_contratos, valor_total AS rapida_valor FROM rapida_ventana_diaria", con)
cmp = batch.merge(rapida, on="fecha", how="outer"); cmp["coinciden"] = (cmp.batch_contratos == cmp.rapida_contratos)
print("Capa batch vs capa rápida, por día:"); cmp

Capa batch vs capa rápida, por día:


,fecha,batch_contratos,batch_valor,rapida_contratos,rapida_valor,coinciden
0,2026-10-25,160,2.260896e+09,160,2.260896e+09,True
1,2026-10-26,2922,3.647818e+11,2922,3.647818e+11,True
2,2026-10-27,2860,1.804518e+12,2860,1.804518e+12,True
3,2026-10-28,3164,9.478867e+11,3164,9.478867e+11,True
4,2026-10-29,3416,1.256261e+12,3416,1.256261e+12,True
5,2026-10-30,4465,1.607311e+12,4465,1.607311e+12,True
6,2026-10-31,465,6.676580e+10,465,6.676580e+10,True
7,2026-11-01,470,1.382078e+10,470,1.382078e+10,True
8,2026-11-02,384,4.559725e+09,384,4.559725e+09,True
9,2026-11-03,3045,9.851108e+11,3045,9.851108e+11,True


## Fase 8 · Zona de servicio, tablero y repositorio
`f8_servicio` carga en **SQLite** los indicadores ya calculados, para que el tablero no haga cálculos: `ind_directa_por_departamento`, `ind_modalidad`, `ind_top_entidades`, `ind_serie_diaria` (+ `ind_cubo_dia_dep_modalidad`, `ind_entidades` y `meta_corte` con fuente y fecha de corte).
El tablero es **Plotly HTML** (una de las herramientas permitidas) y se publica en GitHub Pages: mapa coroplético, barras apiladas, treemap, serie diaria con eje secundario, mapa de calor y tabla con búsqueda; con filtros por fechas, departamento y modalidad.

In [15]:
con = sqlite3.connect(config.SERVICIO_DB)
for t in ["ind_directa_por_departamento", "ind_modalidad", "ind_top_entidades", "ind_serie_diaria", "meta_corte"]:
    n = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"--- {t}: {n} filas"); display(pd.read_sql(f"SELECT * FROM {t} LIMIT 4", con))

--- ind_directa_por_departamento: 34 filas


,departamento,contratos,contratos_directa,valor_total,valor_directa,pct_contratos_directa,pct_valor_directa
0,Distrito Capital de Bogotá,5847,4595,4.809176e+12,5.845934e+11,78.59,12.16
1,Antioquia,3031,2360,1.569318e+12,1.281607e+12,77.86,81.67
2,Atlántico,1456,1305,5.209621e+11,1.226542e+11,89.63,23.54
3,Valle del Cauca,3322,2993,2.358184e+11,9.590483e+10,90.10,40.67


--- ind_modalidad: 8 filas


,modalidad,contratos,valor_total,pct_valor
0,Licitación pública,161,4.421800e+12,51.93
1,Contratación directa,24130,2.557591e+12,30.04
2,Contratación régimen especial,5632,6.212394e+11,7.30
3,Concurso de méritos,133,4.673431e+11,5.49


--- ind_top_entidades: 20 filas


,entidad,departamento,contratos,valor_total,pct_contratos_directa
0,INVIAS,Distrito Capital de Bogotá,129,3.818257e+12,58.91
1,DISTRITO ESPECIAL DE CIENCIA TECNOLOGIA E INNOVACION DE MEDELLIN,Antioquia,35,1.088101e+12,54.29
2,DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA,Atlántico,672,4.857285e+11,99.70
3,FONDO ROTATORIO**,Distrito Capital de Bogotá,24,2.443850e+11,95.83


--- ind_serie_diaria: 14 filas


,fecha_firma,contratos,valor_total,valor_directa
0,2026-10-25,160,2.260896e+09,1.730229e+09
1,2026-10-26,2922,3.647818e+11,1.402426e+11
2,2026-10-27,2860,1.804518e+12,1.337019e+12
3,2026-10-28,3164,9.478867e+11,1.375439e+11


--- meta_corte: 1 filas


,fuente,url,fecha_corte,filas_curada,generado
0,datos.gov.co - SECOP II Contratos Electrónicos (jbjy-vk9h),https://www.datos.gov.co/resource/jbjy-vk9h.json,2026-11-07 23:29:53.311000,32707,2026-11-23T03:49:12Z


In [16]:
from dashboard import generar
generar.main()

tablero generado: docs/index.html (290 KB) · corte 2026-11-07 23:29:53.311000


### Resultado
![Tablero](evidencias/tablero_completo.png)

**Publicado en GitHub Pages:** ver el enlace en el `README.md`. **Evidencias** en `evidencias/` (Kafka, HDFS `ls`, MapReduce vs Spark, idempotencia, streaming tras reinicio, calidad).